In [2]:
# Q1 Embedding a query
import numpy as np
from embedder import Embedder

embed = Embedder()

q = "How does approximate nearest neighbor search work?"

v = embed.encode(q)
print(" The first value v[0] is:", v[0])

 The first value v[0] is: -0.020582034180917443


In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [12]:
doc = next(doc for doc in documents if '07-sqlitesearch-vector' in doc['filename'])

# embed documents
d = embed.encode(doc['content'])

# Q2: Cosine similarity 
score = np.dot(d,v)
print(score)

0.36107008472347096


In [13]:
# Q# chunking and searching by hand 
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Q4: We get {len(chunks)} chunks")
# embbed chunk's content with encode barch 

Q4: We get 295 chunks


In [ ]:
texts = [chunk["content"] for chunk in chunks]

X = embed.encode_batch(texts)

In [33]:
score = X.dot(v)
print(score)

[ 3.15187602e-01  2.01479486e-01  5.90557896e-02 -7.67661873e-02
  1.18452437e-01 -1.41696960e-01 -2.81407287e-02 -4.65670436e-02
 -2.06995100e-02 -6.07744502e-02  2.13273557e-01  8.87600514e-02
 -1.97270699e-02  3.11629861e-01  5.51034657e-01  2.04008050e-01
  2.12515790e-01  1.93648989e-01  2.51961082e-01  1.31078499e-01
  2.59120384e-01  7.63815574e-02  9.59191410e-02  9.81459834e-03
 -3.59108127e-02  1.01210649e-02  1.10786931e-01 -9.90259579e-02
 -3.71171335e-02  7.59056387e-02 -3.35340745e-02  8.86827555e-03
  1.02636483e-01  6.89612636e-02  1.29408770e-01  2.57708937e-01
  3.23680467e-01  1.06350053e-01  5.61891338e-02  2.34017262e-01
  1.97954216e-01  9.64295930e-02  1.93709843e-01  2.16719039e-01
  3.48340254e-01  5.10904824e-02  2.05212751e-01  1.05416092e-01
 -3.25432607e-02  4.94664496e-02  2.38574781e-01 -3.44207339e-02
  1.82165259e-01  2.77928459e-02 -2.26818877e-03  2.91897413e-01
  4.39331234e-02  2.23759495e-01  1.63523474e-01  9.16126080e-02
 -8.39987536e-02  7.88590

In [43]:
idx = np.argmax(score)
idx, score[idx]
idx, score[idx], chunks[idx]


(np.int64(94),
 np.float64(0.6489016216319834),
 {'start': 1000,
  'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies fo

In [38]:
# top 5 
top5 = np.argsort(-score)[:5]
print(top5)

[ 94  14 162  85 253]


In [40]:
for idx in top5: 
    print(score[idx])
    print(chunks[idx])
    print()

0.6489016216319834
{'start': 1000, 'content': 'rch. We score\nthe query against every document and pick the top ones. It always finds\nthe true top matches, but it pays for that by touching everything.\n\nApproximate nearest neighbor (ANN) search takes a shortcut. Instead of\ncomparing against everything, it first narrows down to a region of\nlikely matches. Then it scores only within that region. It may miss the\nabsolute best match, but the results are still good and it\'s much\nfaster.\n\n```text\nNN (exact):    compare query against ALL documents -> top 5\nANN (approx):  narrow down to a region -> compare within region -> top 5\n```\n\n## sqlitesearch\n\nsqlitesearch is the persistent sibling of minsearch, and it solves both\nproblems at once.\n\nWe already used it in module 1 for persistent text search. It also does\nvector search through its `VectorSearchIndex` class. It stores vectors\nin SQLite, a real on-disk database, and uses ANN strategies for\nretrieval. Because the data l

In [79]:
# vector search with minsearch 
from minsearch import VectorSearch

# 1. create an index 
vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

# search 
query = " What metric do we use to evaluate a search engine??" 

query_vector = embed.encode(query)
results = vindex.search(query_vector, num_results=5)
results

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

In [86]:
# Q5: keyword search 
from minsearch import Index

# Step 1: define the structure (what fields, what type)
index = Index(
    text_fields=["content"])

query_compare = "How do I store vectors in PostgreSQL?"

# Step 2: load the actual data
index.fit(chunks)

# search with filters and boosts 
text_results = index.search(query_compare,  num_results=5) 
text_results

[{'start': 4000,
  'content': 'get 0.01.\n\nThe first score for `q1` vs `d` (0.32) is higher, so that query is more\nsimilar to the document about registration. The second score for `q2`\nvs `d` sits near 0, because installing Docker has nothing to do with\nregistration. A score near 0 means the two vectors are about as\ndifferent as they can be.\n\nThat\'s the whole idea behind vector search: similar texts get similar\nvectors, and a dot product tells us how similar.\n\n## Cosine similarity\n\nThe `all-MiniLM-L6-v2` model outputs normalized vectors - vectors with\nunit length. When both vectors are normalized, the dot product equals\ncosine similarity. That\'s why the model documentation says it "uses\ncosine similarity."\n\nCosine similarity measures the angle between two vectors, ignoring\ntheir length:\n\n- 1.0 = same direction (similar)\n- 0.0 = perpendicular (unrelated)\n- -1.0 = opposite direction (opposite meaning)\n\nFormally, if `theta` is the angle between two vectors, cosin

In [87]:
query_vector = embed.encode(query_compare)
vector_results = vindex.search(query_vector, num_results=5)
vector_results

[{'start': 0,
  'content': '# Vector Search with PGVector\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=0P54MFyz-mc&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nMany real databases can do vector search. Elasticsearch has it, and\nthere are dedicated stores like Qdrant and Chroma. We\'ll go with\nPostgres. Most of us already run it at work, and the data engineering\ncourse uses it too. The concept is the same as with sqlitesearch, only\nthe database under the hood changes.\n\n[pgvector](https://github.com/pgvector/pgvector) is the PostgreSQL\nextension that makes this work. Install it and Postgres can do vector\nsimilarity search. On top of that you get the usual production features,\nlike concurrent access, transactions, and large datasets.\n\nWe\'ll run Postgres with pgvector in Docker.\n\n## Starting Postgres with pgvector\n\nPull the image and start a container:\n\n```bash\ndocker run -it \\\n    --name pgvector \\\n    -e POSTGRES_USER=user \\\n    -e POSTGRES_PASSWO

In [84]:
# Q6: Hybrid Search:  Reciprocal Rank Fusion (RRF)
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [91]:
query_rrf = "How do I give the model access to tools?"

# search with filters and boosts 
text_results = index.search(query_rrf,  num_results=5) 

query_vector = embed.encode(query_rrf)
vector_results = vindex.search(query_vector, num_results=5)

results = rrf([vector_results, text_results])

In [92]:
results

[{'start': 4000,
  'content': ' function. `parameters` is a JSON schema\nfor the arguments, and we mark `query` as required so the model always\nfills it in.\n\n## Sending the question with the tool\n\nNow we send the same question as before, but this time we include the\ntool in the request:\n\n```python\nresponse = openai_client.responses.create(\n    model="gpt-5.4-mini",\n    input=messages,\n    tools=[search_tool],\n)\n\nresponse.output\n```\n\nLook at the output. Instead of a message with the answer, the response\ncontains a `function_call` entry. The model decided it needs to search\nthe FAQ before answering. Rather than reply, it asked us to run the\nsearch function first.\n\nLook at the arguments too. The model didn\'t pass our question\nverbatim. It judged the raw question wasn\'t the best query to search\nwith. So it rewrote our enrollment question into search keywords like\n"enroll late join course".\n\n## Executing the function and sending the result back\n\nThe function 

In [5]:
# HOMEWORK - EVALUATION
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

len(documents)

72

In [1]:
# evaluating search 
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [12]:
import os 
import json
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

class Questions(BaseModel): # forced output 
    questions: list[str]

In [18]:
from evaluation_utils import llm_structured_retry

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "content": q,
            "document": doc["filename"]
        })

    return results, usage

In [17]:
doc

{'content': "# Agents\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=6uG4_Ivv60E&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn Part 1 of this module we built RAG pipelines.\n\nEvery pipeline we wrote followed the same flow:\n\n- search the FAQ,\n- build a prompt with the results,\n- send it to the LLM.\n\nThis returns good answers when the user's query matches the documents.\nThe search finds the right entry, the LLM reads it, and you get a\nhelpful reply.\n\nOften, though, the search returns nothing useful.\n\n- Maybe the user made a typo.\n- Maybe they asked the question in an unusual way.\n- Maybe they need information from two different searches.\n\nWe use lexical search here, so the search looks for an exact match.\nOne typo and it misses the entry it needed. In our pipeline there's\nno recovery. The search runs once, and if it returns garbage the LLM\ngets garbage. Our pipeline always does the same thing, no matter what.\n\nInstead of routing the user question str

In [24]:
# first 5 documents only 
from tqdm.auto import tqdm
from evaluation_utils import calc_total_price


ground_truth_3 = []
usages_3 = []

for doc in tqdm(documents[:3]): # first first documents 
    records, usage = generate_ground_truth(doc)
    ground_truth_3.extend(records)
    usages_3.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [27]:
usages_3

[ResponseUsage(input_tokens=1050, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=82, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1132),
 ResponseUsage(input_tokens=1322, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=67, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1389),
 ResponseUsage(input_tokens=1773, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=107, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1880)]

In [25]:
calc_total_price(usages_3)

0.00054299

In [19]:
ground_truth = []
usages = []

for doc in tqdm(documents): # first first documents 
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/72 [00:00<?, ?it/s]